In [3]:
# =============================================================
# Korean 3-Class Selection + HF Tokenization Pipeline
# Google Colab Version
# =============================================================

# -------------------------------------------------------------
# 0. Setup — Colab 패키지 설치
# -------------------------------------------------------------
# Colab 최초 실행 시 아래 셀을 먼저 실행
!pip install -q transformers scikit-learn openpyxl


In [4]:

import os
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
import math

SEED = 42
TOP_K = 10
np.random.seed(SEED)
os.makedirs("figs", exist_ok=True)


# =============================================================
# Part A · Data Selection (3-Class Balanced)
# =============================================================

# 1. Load Dataset — 한국어만 사용
url = "https://raw.githubusercontent.com/hongsukyi/Lectures/main/data/Conversational.xlsx"

df_raw = pd.read_excel(url, engine="openpyxl")
df_raw = df_raw[["원문", "상황"]].dropna()

print(f"Total samples: {len(df_raw):,}")


# 2. Top-K Domain Check
label_counts = df_raw["상황"].value_counts()
top_k = label_counts.head(TOP_K)

print(f"\nTop {TOP_K} Classes")
for i, (label, cnt) in enumerate(top_k.items(), 1):
    print(f"{i:2d}. {label} ({cnt:,})")


# -------------------------------------------------------------
# 3. Class Name Simplification
# -------------------------------------------------------------
label_map = {
    "직장에서의 일상 대화 (상사, 동료, 부하 간의 대화)": "직장",
    "찬성 및 반대": "찬반",
    "취직 면접 상황": "면접",
    "회의 관련": "회의",
    "의견 교환하기": "의견교환",
    "학교생활 (시험, 졸업, 입학 등)": "학교",
    "원하는 스타일에 대해 점원 or 친구와 대화 시술 전/시술 시/시술 후 대화": "시술상담",
    "제안 및 협상하기": "협상",
    "CS/고객 상담 (고객 요청 대응: 취소, 반품, 결함 등)": "고객",
    "마케팅/홍보": "마케팅",
}

df_raw["label_simple"] = df_raw["상황"].map(label_map)

n_unmapped = df_raw["label_simple"].isna().sum()
print(f"\nUnmapped labels: {n_unmapped:,}")

SELECT_LABELS = ["면접", "학교", "협상"]

df_sel = df_raw[df_raw["label_simple"].isin(SELECT_LABELS)].copy()

assert len(df_sel) > 0, \
    f"[ERROR] SELECT_LABELS {SELECT_LABELS} 에 해당하는 데이터가 없습니다."

print("\nSelected Classes:")
for lb in SELECT_LABELS:
    print(f"  - {lb} ({(df_sel['label_simple'] == lb).sum():,})")


# -------------------------------------------------------------
# 4. Balanced Sampling
# -------------------------------------------------------------
SAMP = min(df_sel["label_simple"].value_counts().min(), 800)

df_clf = (
    df_sel.groupby("label_simple")
    .sample(n=SAMP, random_state=SEED)
    .reset_index(drop=True)
)

print("\nBalanced dataset size:", len(df_clf))


# =============================================================
# Part B · HuggingFace Tokenization
# =============================================================

# 5. Prepare Dataset
df = df_clf[["원문", "label_simple"]].copy()
df.rename(columns={"원문": "text", "label_simple": "label"}, inplace=True)

print("Final dataset size:", len(df))
print("-" * 60)


# 6. Load Tokenizer — Colab은 SSL 우회 불필요
MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print("-" * 60)


# -------------------------------------------------------------
# 7. Automatic MAX_LEN (p95 기반)
# -------------------------------------------------------------
lengths = [len(tokenizer.tokenize(t)) for t in df["text"]]
p95, p99 = np.percentile(lengths, [95, 99])

special_tokens_count = tokenizer.num_special_tokens_to_add(pair=False)
margin = 2

raw_max_len = int(p95 + special_tokens_count + margin)
MAX_LEN = int(math.ceil(raw_max_len / 8) * 8)

print(f"p95: {p95:.0f}, p99: {p99:.0f}")
print(f"special tokens: {special_tokens_count}")
print(f"자동 설정 MAX_LEN: {MAX_LEN}")
print("-" * 60)


# -------------------------------------------------------------
# 8. Batch Tokenization
# -------------------------------------------------------------
encoded = tokenizer(
    df["text"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors=None
)

input_ids      = np.array(encoded["input_ids"],      dtype=np.int32)
attention_mask = np.array(encoded["attention_mask"], dtype=np.int32)

pad_id = tokenizer.pad_token_id

print(f"Encoded shape : {input_ids.shape}")
print("-" * 60)


# =============================================================
# Part C · Label Encoding + Single File Save
# =============================================================

# 9. Label Encoding
le = LabelEncoder()
labels = le.fit_transform(df["label"])
class_names = le.classes_.tolist()

print("Class names:", class_names)


# 10. Save — 단일 파일 완결형
n_total = len(df_clf)
fname = f"aihub_enko_conv_kluebert_3class_n{n_total}_v1.npz"

np.savez(
    fname,
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    vocab_size=np.int32(tokenizer.vocab_size),
    max_len=np.int32(MAX_LEN),
    pad_id=np.int32(pad_id),
    num_classes=np.int32(len(class_names)),
    class_names=np.array(class_names)
)

print(f"\nSaved: {fname}")


# -------------------------------------------------------------
# 11. Google Drive 저장 (선택)
# -------------------------------------------------------------
# Colab 세션 종료 시 파일이 삭제되므로 Drive 저장 권장
#
# from google.colab import drive
# drive.mount("/content/drive")
#
# import shutil
# shutil.copy(fname, f"/content/drive/MyDrive/{fname}")
# print(f"Drive 저장 완료: /content/drive/MyDrive/{fname}")


# -------------------------------------------------------------
# [참고] 로드 방법
# -------------------------------------------------------------
# data = np.load(fname, allow_pickle=True)
#
# input_ids      = data["input_ids"]
# attention_mask = data["attention_mask"]
# labels         = data["labels"]
# vocab_size     = data["vocab_size"].item()
# max_len        = data["max_len"].item()
# pad_id         = data["pad_id"].item()
# num_classes    = data["num_classes"].item()
# class_names    = data["class_names"].tolist()

Total samples: 100,000

Top 10 Classes
 1. 직장에서의 일상 대화 (상사, 동료, 부하 간의 대화) (1,320)
 2. 찬성 및 반대 (1,076)
 3. 취직 면접 상황 (1,032)
 4. 회의 관련 (1,020)
 5. 의견 교환하기 (1,016)
 6. 학교생활 (시험, 졸업, 입학 등) (976)
 7. 원하는 스타일에 대해 점원 or 친구와 대화 시술 전/시술 시/시술 후 대화 (948)
 8. 제안 및 협상하기 (924)
 9. CS/고객 상담 (고객 요청 대응: 취소, 반품, 결함 등) (832)
10. 마케팅/홍보 (804)

Unmapped labels: 90,052

Selected Classes:
  - 면접 (1,032)
  - 학교 (976)
  - 협상 (924)

Balanced dataset size: 2400
Final dataset size: 2400
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Tokenizer: klue/bert-base
Vocab size: 32,000
------------------------------------------------------------
p95: 25, p99: 30
special tokens: 2
자동 설정 MAX_LEN: 32
------------------------------------------------------------
Encoded shape : (2400, 32)
------------------------------------------------------------
Class names: ['면접', '학교', '협상']

Saved: aihub_enko_conv_kluebert_3class_n2400_v1.npz
